In [ ]:
# system tools
import sys
from pathlib import Path

# battle processing
import json
from zipfile import ZipFile
from tools.battle import Battle
from tools.full_pokemon import FullPokemon
import copy

# data science
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score,StratifiedKFold,train_test_split
from sklearn.base import BaseEstimator,ClassifierMixin
from sklearn.metrics import accuracy_score,confusion_matrix
import statsmodels.api as sm

repo = Path.cwd().resolve()
sys.path.append(str(repo / "damage-calc-python-wrapper"))
sys.path.append(str(repo / "damage-calc-python-wrapper" / "python_calc"))

num_training_zips = 3 # change if you want fewer/more battles at the benefit/cost of less/more time; current max is 3
replay_dir = repo / "data" / "replays"
zip_paths = [replay_dir / f"gen9randombattles_{i}.zip" for i in range(1,num_training_zips+1)]
replay_zips = [ZipFile(zip_path,'r') for zip_path in zip_paths]

from python_calc import (
    Pokemon,
    Move,
    Field,
    Side,
    calc,
    advantage
)

In [2]:
class BaselineEloPredictor(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"
    coef_ = np.array([[np.log(10)/400]])

    def __init__(self, scale=400):
        self.scale = scale

    def fit(self, X, y,offset = None): #offset should not be used, it's merely there to have the same signature as log-reg-with-offset.fit
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        return self

    def predict_proba(self, X,offset = None):
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        elo_diff = X[:, 0].astype(float)
        p = 1 / (1 + 10 ** (-elo_diff / self.scale))
        return np.column_stack([1 - p, p])

    def predict(self, X, offset = None):
        probs = self.predict_proba(X)[:, 1]
        preds = (probs >= 0.5).astype(int)
        return self.classes_[preds]

    # def score(self, X, y):
    #     return np.mean(self.predict(X) == y)

    # def __sklearn_tags__(self):
    #     tags = super().__sklearn_tags__()
    #     tags.estimator_type = "classifier"
    #     return tags

In [3]:
class LogisticRegressionWithOffset(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def fit(self, X, y, offset=None):
        self.offset = offset
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]


        self.fitted = sm.GLM(y, X, family=sm.families.Binomial(), offset=offset).fit()
        self.coef_ = np.array([self.fitted.params])
        return self

    def predict_proba(self, X,offset = None):
        p = np.array(self.fitted.predict(X, offset=offset)).reshape(-1, 1)
        return np.concatenate([1 - p, p], axis=1)

    def predict(self, X,offset = None):
        return 1*(self.predict_proba(X,offset=offset)[:,1]>=0.5)


## Example Calculations

In [4]:
palafin = Pokemon(
    name="Palafin-Hero",
    gen=9,
    level=100,
    moves=["Bulk Up", "Overheat","Wave Crash","Drain Punch"],
    evs={"hp" : 248, "atk" : 8, "spd" : 252},
    nature="Careful",
    item="Leftovers")

In [5]:
delphox = Pokemon(
    gen=9,
    name="Delphox",
    evs={"spa" : 252, "spe" : 252, "hp" : 4},
    nature = "Jolly",
    moves = ["fireblast"]
)

In [6]:
gengar = Pokemon(
    gen=9,
    name="Gengar",
    level=100,
    item="Choice Specs",
    nature="Timid",
    evs={"spa" : 252, "spe" : 252, "spd" : 4},
    moves = ["Shadow Ball", "Sludge Wave", "Thunderbolt", "Focus Blast"],
    curHP=261)

In [7]:
tauros_p_a = Pokemon(
    name="taurospaldeaaqua",
    gen=9,
    level=100
)

In [8]:
fire_blast = Move(gen=9,name="fireblast")

In [9]:
calc.calculate(gen = 9, attacker = delphox, defender = tauros_p_a, move = fire_blast,field=Field())

{'gen': 9,
 'attacker': {'name': 'Delphox',
  'ability': 'Blaze',
  'item': None,
  'level': 100,
  'nature': 'Jolly',
  'types': ['Fire', 'Psychic'],
  'stats': {'hp': 292,
   'atk': 174,
   'def': 180,
   'spa': 294,
   'spd': 236,
   'spe': 337},
  'rawStats': {'hp': 292,
   'atk': 174,
   'def': 180,
   'spa': 294,
   'spd': 236,
   'spe': 337},
  'boosts': {'hp': 0, 'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0},
  'originalCurHP': 292,
  'isDynamaxed': None,
  'teraType': None,
  'status': '',
  'toxicCounter': 0},
 'defender': {'name': 'taurospaldeaaqua',
  'ability': 'Intimidate',
  'item': None,
  'level': 100,
  'nature': 'Serious',
  'types': ['Fighting', 'Water'],
  'stats': {'hp': 291,
   'atk': 256,
   'def': 246,
   'spa': 96,
   'spd': 176,
   'spe': 236},
  'rawStats': {'hp': 291,
   'atk': 256,
   'def': 246,
   'spa': 96,
   'spd': 176,
   'spe': 236},
  'boosts': {'hp': 0, 'atk': 0, 'def': 0, 'spa': 0, 'spd': 0, 'spe': 0},
  'originalCurHP': 291,
  'isDynamaxed':

In [ ]:
id = "2631906096"
with open("data/replays/gen9-randombattle/gen9randombattle-" + id + ".json") as battle_json:
    data = json.load(battle_json)

team1 = [
    Pokemon(
        name=data["teams_full"][0][mon_name]["speciesId"],
        gen=9,
        level=data["teams_full"][0][mon_name]["level"],
        ability = data["teams_full"][0][mon_name]["ability"],
        item = data["teams_full"][0][mon_name]["item"],
        gender = data["teams_full"][0][mon_name]["gender"],
        ivs = data["teams_full"][0][mon_name]["ivs"],
        evs = data["teams_full"][0][mon_name]["evs"],
        # teraType = data["teams_full"][0][mon_name]["teraType"],
        moves = data["teams_full"][0][mon_name]["moves"]
    )
    for mon_name in data["teams_full"][0].keys()
]

team2 = [
    Pokemon(
        name=data["teams_full"][1][mon_name]["speciesId"],
        gen=9,
        level=data["teams_full"][1][mon_name]["level"],
        ability = data["teams_full"][1][mon_name]["ability"],
        item = data["teams_full"][1][mon_name]["item"],
        gender = data["teams_full"][1][mon_name]["gender"],
        ivs = data["teams_full"][1][mon_name]["ivs"],
        evs = data["teams_full"][1][mon_name]["evs"],
        # teraType = data["teams_full"][1][mon_name]["teraType"], # calculate will assume that the teraType is on
        moves = data["teams_full"][1][mon_name]["moves"]
    )
    for mon_name in data["teams_full"][1].keys()
]

In [11]:
rows = [[team1[i].name] + [advantage(m1=team1[i],m2=team2[j],capped=False) for j in range(6)] for i in range(6)]
df = pd.DataFrame(rows,columns=['team1'] + [team2[j].name for j in range(6)])
df

,team1,abomasnow,ceruledge,chansey,hippowdon,carbink,klefki
0,quaquaval,1.347997,1.079847,1.402815,0.727937,0.539845,0.778549
1,pecharunt,0.722992,0.410621,1.000000,-0.646996,0.464214,0.463190
2,noivern,-0.919896,0.535204,-0.706745,-0.216687,-0.892374,0.451061
3,azumarill,0.277500,1.039758,0.936618,1.249278,1.111456,0.794242
4,gogoat,-0.900685,-0.637954,0.623522,0.912558,0.787418,0.850608
5,klawf,1.267300,1.180102,0.706782,-0.825678,-0.217348,0.972550


## Comparing new and old

In [12]:
# The list of traits and the method names in FullPokemon which return True if and only if the Pokemon has the desired trait
trait_method_map = {
        "trappers": "is_trapper",
        "type_changers": "is_type_changer",
        "weather_setters": "is_weather_setter",
        "terrain_setters": "is_terrain_setter",
        "stat_drop_resistors": "is_stat_drop_resistor",
        "absorbers": "is_absorber",
        "extra_immunities": "has_extra_immunities",
        "status_resists": "is_status_resistor",
        "contact_punishers": "is_contact_punisher",
        "ability_ignorers": "is_ability_ignorer",
        "pranksters": lambda mon: mon.ability == "Prankster",
        "intimidators": lambda mon: mon.ability == "Intimidate",
        "unaware": lambda mon: mon.ability == "Unaware",
        "harvest": lambda mon: mon.ability == "Harvest",
        "regenerators": lambda mon: mon.ability == "Regenerator",
        "serene_grace": lambda mon: mon.ability == "Serene Grace",
        "sturdy": lambda mon: mon.ability == "Sturdy",
        "illusion": lambda mon: mon.ability == "Illusion",
        "triage": lambda mon: mon.ability == "Triage",
        "boosting_abilities": "has_boosting_ability",
        "weather_boosters": "is_weather_booster",
        "omni_boosters": "has_omni_boost",
        "off_def_spe_boosters": "has_off_def_spe_boost",
        "off_spe_boosters": "has_off_spe_boost",
        "off_def_boosters": "has_off_def_boost",
        "off_boosters": "has_off_boost",
        "spe_boosters": "has_spe_boost",
        "def_boosters": "has_def_boost",
        "move_boosters": "has_boost_move",
}

traits = trait_method_map.keys()

# Given a trait and a team of Pokemon, returns the number of Pokemon on that team with the trait
def count_trait(team, trait):
        method = trait_method_map[trait]
        if isinstance(method, str):
                return sum(int(getattr(mon, method)()) for mon in team)
        return sum(int(method(mon)) for mon in team)

In [ ]:
# This is for making the csv.  It takes about 10 minutes on my computer.  Ignore this cell and use the next cell to load the csv.

# files = [replay_zip.read(file_name) for replay_zip in replay_zips for file_name in replay_zip.namelist()]
# rows = []

# for file in files:
#     try:
#         data = json.loads(file)
#         battle = Battle(data_json=data,parse=True)
#         if not battle.custom_ruleQ:

#             row_dict = {
#                         "id": battle.id,
#                         "p1": battle.players[0],
#                         "p2": battle.players[1],
#                         "duration": battle.end_time - battle.start_time,
#                         "p1_rating" : battle.player_dets[0]["rating"],
#                         "elo_diff": battle.player_dets[0]["rating"] - battle.player_dets[1]["rating"],
#                         "p1_wins" : battle.players[0] == battle.winner.name,
#                         "p1_revealed_team_size" : len(battle.teams[0].keys()),
#                         "p2_revealed_team_size" : len(battle.teams[1].keys())
#                     }

#             # old advantage calcs
#             team1 = [FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
#             team2 = [FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]

#             for m1 in range(6):
#                 for m2 in range(6):
#                     row_dict[f"p1_m{m1}_old_adv_over_p2_m{m2}"] = FullPokemon.advantage(m1=team1[m1],m2=team2[m2])

#             row_dict["old_adv"] = sum(row_dict[f"p1_m{m1}_old_adv_over_p2_m{m2}"] for m1 in range(6) for m2 in range(6))
            
#             # trait diffs
#             for trait in traits:
#                 p1_num_trait = count_trait(team1, trait)
#                 p2_num_trait = count_trait(team2, trait)
#                 row_dict[f"num_{trait}_diff"] = p1_num_trait - p2_num_trait

#             # new advantage calcs
#             team1 = [
#                 Pokemon(
#                     name=data["teams_full"][0][mon_name]["speciesId"],
#                     gen=9,
#                     level=data["teams_full"][0][mon_name]["level"],
#                     ability = data["teams_full"][0][mon_name]["ability"],
#                     item = data["teams_full"][0][mon_name]["item"],
#                     gender = data["teams_full"][0][mon_name]["gender"],
#                     ivs = data["teams_full"][0][mon_name]["ivs"],
#                     evs = data["teams_full"][0][mon_name]["evs"],
#                     # teraType = data["teams_full"][0][mon_name]["teraType"],
#                     moves = data["teams_full"][0][mon_name]["moves"]
#                 )
#                 for mon_name in data["teams_full"][0].keys()
#             ]

#             team2 = [
#                 Pokemon(
#                     name=data["teams_full"][1][mon_name]["speciesId"],
#                     gen=9,
#                     level=data["teams_full"][1][mon_name]["level"],
#                     ability = data["teams_full"][1][mon_name]["ability"],
#                     item = data["teams_full"][1][mon_name]["item"],
#                     gender = data["teams_full"][1][mon_name]["gender"],
#                     ivs = data["teams_full"][1][mon_name]["ivs"],
#                     evs = data["teams_full"][1][mon_name]["evs"],
#                     # teraType = data["teams_full"][1][mon_name]["teraType"], # calculate will assume that the teraType is on
#                     moves = data["teams_full"][1][mon_name]["moves"]
#                 )
#                 for mon_name in data["teams_full"][1].keys()
#             ]
    
#             for m1 in range(6):
#                 for m2 in range(6):
#                     row_dict[f"p1_m{m1}_new_capped_adv_over_p2_m{m2}"] = advantage(m1=team1[m1],m2=team2[m2],capped=True)
#                     row_dict[f"p1_m{m1}_new_uncapped_adv_over_p2_m{m2}"] = advantage(m1=team1[m1],m2=team2[m2],capped=False)

#             row_dict["new_capped_adv"] = sum(row_dict[f"p1_m{m1}_new_capped_adv_over_p2_m{m2}"] for m1 in range(6) for m2 in range(6))
#             row_dict["new_uncapped_adv"] = sum(row_dict[f"p1_m{m1}_new_uncapped_adv_over_p2_m{m2}"] for m1 in range(6) for m2 in range(6))
        
#             rows.append(row_dict)
#     except (json.JSONDecodeError,UnicodeDecodeError):
#         continue

# full_match_data = pd.DataFrame(rows)

# elo_diff_coef = np.log(10) / 400
# full_match_data["elo_diff_offset"] = elo_diff_coef * full_match_data["elo_diff"]

# full_match_data.to_csv(path_or_buf= repo / "data" / "new_adv_data.csv")

In [14]:
full_match_data = pd.read_csv(repo / "data" / "new_adv_data.csv").drop('Unnamed: 0',axis=1)
full_match_data

,id,p1,p2,duration,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,p1_m0_old_adv_over_p2_m0,...,p1_m5_new_uncapped_adv_over_p2_m2,p1_m5_new_capped_adv_over_p2_m3,p1_m5_new_uncapped_adv_over_p2_m3,p1_m5_new_capped_adv_over_p2_m4,p1_m5_new_uncapped_adv_over_p2_m4,p1_m5_new_capped_adv_over_p2_m5,p1_m5_new_uncapped_adv_over_p2_m5,new_capped_adv,new_uncapped_adv,elo_diff_offset
0,gen9randombattle-2631906096,sufideu,saberclaw,598,1135,-5,False,6,5,1.147629,...,0.706782,-0.574759,-0.825678,-0.176907,-0.217348,0.646369,0.972550,11.630022,16.669602,-0.028782
1,gen9randombattle-2631763570,PineappleCats,L4V,167,1959,10,False,6,6,-0.826425,...,0.352018,-0.286680,-0.530582,1.000000,1.175305,0.471799,0.977149,-1.459568,-1.810890,0.057565
2,gen9randombattle-2631369343,Chicken347,cococem,275,1999,-69,True,6,6,0.328214,...,1.071262,-0.413013,-0.515417,0.365865,0.599888,-0.270325,-0.426094,6.683373,11.314115,-0.397196
3,gen9randombattle-2631529004,WhatEver2102,Duck Cop,123,1999,17,True,1,3,-0.173456,...,0.816808,0.000000,-0.036792,0.382589,0.428044,0.451786,0.620654,4.304449,5.925810,0.097860
4,gen9randombattle-2631993792,monomythic,OverthereStair,301,2120,58,False,6,6,-0.290351,...,0.416380,0.325850,0.767722,0.179207,0.327515,1.000000,1.127956,-0.922874,-1.283986,0.333875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12761,gen9randombattle-2642081603,datboi480,throwaway20some,436,1933,-68,False,6,6,0.715345,...,-1.000170,0.346467,0.404160,1.000000,1.000000,-0.098242,-0.579900,-1.340627,-4.877436,-0.391439
12762,gen9randombattle-2641877736,testbotcif,lay88,239,1325,-8,False,6,6,-0.297292,...,-0.743068,-0.571055,-1.432727,-0.430018,-0.452892,0.174352,0.741621,-2.306565,-3.276725,-0.046052
12763,gen9randombattle-2642095210,notmetbh102,Uday30,187,2220,-49,False,6,5,-0.521765,...,1.944737,-0.125484,-0.193937,0.535714,0.668896,0.507688,0.843598,4.724409,7.833947,-0.282067
12764,gen9randombattle-2642125538,Alienwere,forcemajor14,186,2023,4,False,6,5,1.048859,...,-0.639468,0.065543,0.497912,-0.174519,-0.369744,0.174157,0.219272,-4.298336,-5.438579,0.023026


In [15]:
# We should throw away matches where people rage quit early
complete_matches = full_match_data[(full_match_data['duration'] > 60) & ((full_match_data["p1_revealed_team_size"] > 2) | (full_match_data["p2_revealed_team_size"] > 2))]
# This is to grab matches where we know that the players understand the basic switching strategy (from Marz' work on switching), 1965 is standard
threshold = 1965
highly_rated_matches = complete_matches[(complete_matches['p1_rating'] > threshold) & (complete_matches[['p1_rating','elo_diff']].sum(axis=1) > threshold)]

In [16]:
highly_rated_matches

,id,p1,p2,duration,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,p1_m0_old_adv_over_p2_m0,...,p1_m5_new_uncapped_adv_over_p2_m2,p1_m5_new_capped_adv_over_p2_m3,p1_m5_new_uncapped_adv_over_p2_m3,p1_m5_new_capped_adv_over_p2_m4,p1_m5_new_uncapped_adv_over_p2_m4,p1_m5_new_capped_adv_over_p2_m5,p1_m5_new_uncapped_adv_over_p2_m5,new_capped_adv,new_uncapped_adv,elo_diff_offset
3,gen9randombattle-2631529004,WhatEver2102,Duck Cop,123,1999,17,True,1,3,-0.173456,...,0.816808,0.000000,-0.036792,0.382589,0.428044,0.451786,0.620654,4.304449,5.925810,0.097860
4,gen9randombattle-2631993792,monomythic,OverthereStair,301,2120,58,False,6,6,-0.290351,...,0.416380,0.325850,0.767722,0.179207,0.327515,1.000000,1.127956,-0.922874,-1.283986,0.333875
7,gen9randombattle-2631439736,Mr Brightside,indias last hope,448,2115,-144,False,6,6,-0.771241,...,0.332161,-0.400781,-0.892161,0.397270,1.084770,0.134100,0.766147,7.048926,10.275611,-0.828931
8,gen9randombattle-2631771408,Illuminating_Fate,medo6037,287,2170,-7,True,5,4,1.150289,...,-0.239610,0.287273,0.472574,-0.020522,-0.115522,-0.419913,-0.625368,-1.293582,-4.767283,-0.040295
14,gen9randombattle-2631594339,szbsb,Bigoleg,417,2047,-36,True,5,6,0.357055,...,1.346962,0.750000,1.051754,-0.586106,-0.951125,0.353137,0.368319,1.553845,5.587282,-0.207233
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12749,gen9randombattle-2642114009,forcemajor14,majex,196,2084,2,False,6,6,0.612786,...,1.345191,0.779797,0.931090,-0.316978,-0.471048,-0.483209,-1.406659,-0.687987,2.770689,0.011513
12756,gen9randombattle-2642143701,forcemajor14,lexam22,246,2235,-12,False,6,1,-0.686908,...,-0.115547,0.201961,0.496961,0.386765,1.180579,-0.396698,-1.362384,5.259238,5.940640,-0.069078
12759,gen9randombattle-2641938445,andonibavi,qiuescent,277,2173,-43,True,5,5,-0.788750,...,-0.444012,0.134829,0.369204,0.247774,0.322099,-0.429816,-0.536640,-5.738979,-8.897955,-0.247528
12763,gen9randombattle-2642095210,notmetbh102,Uday30,187,2220,-49,False,6,5,-0.521765,...,1.944737,-0.125484,-0.193937,0.535714,0.668896,0.507688,0.843598,4.724409,7.833947,-0.282067


In [17]:
# Let's check the p-values of each individual feature in a logistic regression
features = ['new_capped_adv', 'new_uncapped_adv', 'old_adv']
df = complete_matches # good options include: complete_matches, highly_rated_matches

models = [sm.Logit(df['p1_wins'],df[[feature]]).fit(disp=False) for feature in features]

rows = []
for i in range(len(features)):
    summary = pd.Series({"feature" : features[i], "p-value" : models[i].pvalues.iloc[0], "coefficient" : models[i].params.iloc[0]})
    rows.append(summary)

table = pd.DataFrame(rows)
table.sort_values("p-value",ascending=True)

,feature,p-value,coefficient
1,new_uncapped_adv,4.983477e-13,0.015289
0,new_capped_adv,1.822187e-12,0.024866
2,old_adv,1.465210e-10,0.018167


In [ ]:
n_splits = 10
lr = LogisticRegression(C=np.inf,fit_intercept=False,random_state=207,max_iter=1000)
skf = StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=207)

model_info = [
    # tuples of the form (model_name, model, X_train, offset)
    ('new_capped_adv_only',LogisticRegressionWithOffset(),df[["new_capped_adv"]],None),
    ('new_uncapped_adv_only',LogisticRegressionWithOffset(),df[["new_uncapped_adv"]],None),
    ('old_adv_only',LogisticRegressionWithOffset(),df[["old_adv"]],None),
    ('elo_diff_only',BaselineEloPredictor(),df[["elo_diff"]],None),
    ('new_uncapped_adv_elo_diff',LogisticRegressionWithOffset(),df[["new_uncapped_adv"]],df["elo_diff_offset"]),
    ('new_capped_adv_elo_diff', LogisticRegressionWithOffset(), df[["new_capped_adv"]], df["elo_diff_offset"]),
    ('old_adv_elo_diff',LogisticRegressionWithOffset(),df[["old_adv"]],df["elo_diff_offset"])
]

model_accs = np.zeros(shape=(len(model_info),n_splits))

for model_index,(model_name,model,data_set,offset) in enumerate(model_info):
    for fold_index,(train_indices,test_indices) in enumerate(skf.split(X = df,y=df["p1_wins"])):
        # set up train-train sets
        X_tt = data_set.iloc[train_indices]
        y_tt = df.iloc[train_indices]["p1_wins"]
        if isinstance(offset,pd.Series):
            offset_tt = offset.iloc[train_indices]
        else:
            offset_tt = None

        # set up validation sets
        X_val = data_set.iloc[test_indices]
        y_val = df.iloc[test_indices]["p1_wins"]
        if isinstance(offset,pd.Series):
            offset_val = offset.iloc[test_indices]
        else:
            offset_val = None

        # fit and predict
        model.fit(X=X_tt,y=y_tt,offset = offset_tt)
        preds = model.predict(X=X_val,offset=offset_val)
        model_accs[model_index,fold_index] = accuracy_score(y_true=y_val,y_pred=preds)
    cv_scores = model_accs[model_index,:]
    print(f"The average accuracy score for the {model_name} model is {100*np.mean(cv_scores):.2f} +/- {100*np.std(cv_scores,ddof=1):.2f}%.")
    print(f"On the last fold, the coefficients for this model were {model.coef_[0][0]:.4f}.")
    print()

The average accuracy score for the new_capped_adv_only model is 52.54 +/- 1.63%.
On the last fold, the coefficients for this model were 0.0260.

The average accuracy score for the new_uncapped_adv_only model is 52.82 +/- 1.28%.
On the last fold, the coefficients for this model were 0.0157.

The average accuracy score for the old_adv_only model is 52.39 +/- 0.73%.
On the last fold, the coefficients for this model were 0.0181.

The average accuracy score for the elo_diff_only model is 52.17 +/- 0.91%.
On the last fold, the coefficients for this model were 0.0058.

The average accuracy score for the new_uncapped_adv_elo_diff model is 53.04 +/- 0.98%.
On the last fold, the coefficients for this model were 0.0165.

The average accuracy score for the new_capped_adv_elo_diff model is 53.07 +/- 1.06%.
On the last fold, the coefficients for this model were 0.0270.

The average accuracy score for the old_adv_elo_diff model is 53.01 +/- 1.14%.
On the last fold, the coefficients for this model wer

In [22]:
features = [f"num_{trait}_diff" for trait in traits]

models = [sm.Logit(df['p1_wins'],df[['new_capped_adv',feature]],offset=df['elo_diff_offset']).fit(disp=False) for feature in features]

rows = []
for i in range(len(features)):
    summary = pd.Series({"feature" : features[i], "p-value" : models[i].pvalues.iloc[1], "coefficient" : models[i].params.iloc[1]})
    rows.append(summary)

table = pd.DataFrame(rows)
table.sort_values("p-value",ascending=True)

,feature,p-value,coefficient
18,num_triage_diff,0.011227,0.268617
20,num_weather_boosters_diff,0.017281,-0.077360
2,num_weather_setters_diff,0.022836,0.080935
28,num_move_boosters_diff,0.026361,0.024568
26,num_spe_boosters_diff,0.050532,0.028917
27,num_def_boosters_diff,0.058863,0.027938
23,num_off_spe_boosters_diff,0.063467,0.035235
25,num_off_boosters_diff,0.084886,0.019433
10,num_pranksters_diff,0.132389,0.060004
24,num_off_def_boosters_diff,0.187864,0.020367


In [ ]:
n_splits = 10
lr = LogisticRegression(C=np.inf,fit_intercept=False,random_state=207,max_iter=1000)
skf = StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=207)

model_info = [
    # tuples of the form (model_name, model, X_train, offset)
    ('baseline', LogisticRegressionWithOffset(), df[["new_capped_adv"]], df["elo_diff_offset"]),
    ('triage',LogisticRegressionWithOffset(),df[["new_capped_adv","num_triage_diff"]],df["elo_diff_offset"]),
    ('weather_boosters',LogisticRegressionWithOffset(),df[["new_capped_adv","num_weather_boosters_diff"]],df["elo_diff_offset"]),
    ('weather_setters',LogisticRegressionWithOffset(),df[["new_capped_adv","num_weather_setters_diff"]],df["elo_diff_offset"]),
    ('move_boosters',LogisticRegressionWithOffset(),df[["new_capped_adv","num_move_boosters_diff"]],df["elo_diff_offset"]),
    ('ability_boosters',LogisticRegressionWithOffset(),df[["new_capped_adv","num_boosting_abilities_diff"]],df["elo_diff_offset"])
]

model_accs = np.zeros(shape=(len(model_info),n_splits))

for model_index,(model_name,model,data_set,offset) in enumerate(model_info):
    for fold_index,(train_indices,test_indices) in enumerate(skf.split(X = df,y=df["p1_wins"])):
        # set up train-train sets
        X_tt = data_set.iloc[train_indices]
        y_tt = df.iloc[train_indices]["p1_wins"]
        if isinstance(offset,pd.Series):
            offset_tt = offset.iloc[train_indices]
        else:
            offset_tt = None

        # set up validation sets
        X_val = data_set.iloc[test_indices]
        y_val = df.iloc[test_indices]["p1_wins"]
        if isinstance(offset,pd.Series):
            offset_val = offset.iloc[test_indices]
        else:
            offset_val = None

        # fit and predict
        model.fit(X=X_tt,y=y_tt,offset = offset_tt)
        preds = model.predict(X=X_val,offset=offset_val)
        model_accs[model_index,fold_index] = accuracy_score(y_true=y_val,y_pred=preds)
    
    cv_scores = model_accs[model_index,:]
    print(f"The average accuracy score for the {model_name} model is {100*np.mean(cv_scores):.2f} +/- {100*np.std(cv_scores,ddof=1):.2f}%.")
    print(f"On the last fold, the coefficients for this model were {model.coef_}.")
    cm = confusion_matrix(y_val,preds,normalize='all')
    print(f"The confusion matrix for the {model_name} model is")
    print(cm)
    print()

The average accuracy score for the baseline model is 53.07 +/- 1.06%.
On the last fold, the coefficients for this model were [[0.02700151]].
The confusion matrix for the baseline model is
[[0.31136738 0.21004942]
 [0.27100494 0.20757825]]

The average accuracy score for the triage model is 53.06 +/- 1.00%.
On the last fold, the coefficients for this model were [[0.02798593 0.29667499]].
The confusion matrix for the triage model is
[[0.30807249 0.21334432]
 [0.26853377 0.21004942]]

The average accuracy score for the weather_boosters model is 53.16 +/- 1.02%.
On the last fold, the coefficients for this model were [[ 0.02739005 -0.08782853]].
The confusion matrix for the weather_boosters model is
[[0.30642504 0.21499176]
 [0.27100494 0.20757825]]

The average accuracy score for the weather_setters model is 52.98 +/- 1.01%.
On the last fold, the coefficients for this model were [[0.02727882 0.07558331]].
The confusion matrix for the weather_setters model is
[[0.31136738 0.21004942]
 [0.27

In [ ]:
# look at calibration for new v old
# look at confusion matrices for new v old
# look into calibration in-the-large
# look into range of predict_proba outcomes
# include num_hazard_setters_diff as a features